# ⚙️ Feature Engineering Pipeline

This notebook demonstrates the data cleaning and feature engineering steps for the Smart Pricing Engine.

In [ ]:
import sys
import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

from src.utils.config import *
from src.data.data_pipeline import DataPipeline
from src.data.feature_engineering import FeatureEngineer

## 1. Load Raw Data
We start by loading the raw generated sales data.

In [ ]:
data_path = RAW_DATA_DIR / 'sales_data.csv'
df = pd.read_csv(data_path)
print(f"Raw data shape: {df.shape}")
display(df.head())

## 2. Data Cleaning
The data pipeline handles type conversions and sorts values.

In [ ]:
pipeline = DataPipeline()
print("Before cleaning info:")
df.info()

df_clean = pipeline.clean_data(df)
print("\nAfter cleaning info:")
df_clean.info()

## 3. Feature Engineering
We create additional features necessary for modeling, like rolling averages and lag features.

In [ ]:
engineer = FeatureEngineer()
df_features = engineer.create_features(df_clean)
print(f"Data shape after feature engineering: {df_features.shape}")
print("\nNew columns added:")
print(list(set(df_features.columns) - set(df_clean.columns)))
display(df_features.head())

## 4. Feature Descriptions

| Feature Name | Description |
| :--- | :--- |
| `price_to_cost_ratio` | Ratio of selling price to cost price |
| `price_vs_competitor` | Difference between our price and competitor price |
| `sales_lag_7d` | Units sold 7 days ago |
| `sales_lag_14d` | Units sold 14 days ago |
| `sales_lag_30d` | Units sold 30 days ago |
| `sales_rolling_7d_mean` | Average sales over the last 7 days |
| `sales_rolling_14d_mean` | Average sales over the last 14 days |
| `sales_rolling_30d_mean` | Average sales over the last 30 days |
| `price_rolling_7d_mean` | Average price over the last 7 days |
| `days_to_next_festival` | Number of days until the next festival |
| `log_price` | Natural logarithm of selling price |
| `log_demand` | Natural logarithm of units sold |

## 5. Feature Distributions
Let's look at the distributions of some key newly engineered features.

In [ ]:
fig1 = px.histogram(df_features, x="price_to_cost_ratio", title="Distribution of Price to Cost Ratio")
fig1.show()

fig2 = px.histogram(df_features, x="price_vs_competitor", title="Distribution of Price vs Competitor")
fig2.show()

fig3 = px.histogram(df_features, x="sales_rolling_7d_mean", title="Distribution of 7-Day Rolling Mean Sales")
fig3.show()

## 6. Feature Correlations
Let's observe how these numerical features correlate with the target variable (`units_sold`).

In [ ]:
numeric_cols = df_features.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = df_features[numeric_cols].corr()
target_corr = corr_matrix[TARGET_COLUMN].sort_values(ascending=False)
print(f"Correlation with target ({TARGET_COLUMN}):")
display(target_corr)

fig = px.bar(x=target_corr.index, y=target_corr.values, title="Correlation of features with Units Sold")
fig.show()

## 7. Train/Val/Test Split
We perform a chronological split to evaluate the model correctly on unseen future data.

In [ ]:
train_df, val_df, test_df = pipeline.split_data(df_features)

print(f"Train size: {len(train_df)} ({len(train_df)/len(df_features)*100:.1f}%)")
print(f"Val size: {len(val_df)} ({len(val_df)/len(df_features)*100:.1f}%)")
print(f"Test size: {len(test_df)} ({len(test_df)/len(df_features)*100:.1f}%)\n")

print(f"Train dates: {train_df['date'].min()} to {train_df['date'].max()}")
print(f"Val dates: {val_df['date'].min()} to {val_df['date'].max()}")
print(f"Test dates: {test_df['date'].min()} to {test_df['date'].max()}")

## 8. Feature Selection
We use a defined set of features for training the ML model.

In [ ]:
print("Features selected for ML model:")
for feature in ALL_FEATURES:
    print(f"- {feature}")